# ==============================================
# ASSIGNMENT 2: An Investigation of the Components of Pre-Training and Post-Training in LLMs
# ==============================================

# Instructions:
## 1. Use Google Colab for all experiments (free GPU tier is sufficient).
## 2. This notebook provides a complementory solution for all parts of the assignment.

# ==============================================
# SUBMISSION INSTRUCTION
# ==============================================

## 1. Please write the name of the file as `Group_(number)_assignemnt_2_solution.ipynb`

##2. Only one member from one group needs to submit the solution, to avoid any duplicasy.

## **Question 1: Analyzing with or without Adapter fine-tuning for Multi-Document Summarization (MDS) task.**

## PART 1, 2 and 3

In [ ]:
# Force a clean environment by removing old cache and reinstalling a stable version
!rm -rf /root/.cache/huggingface/datasets
!pip install -q datasets==2.18.0

# Set a safe cache location to avoid file system errors
import os
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

In [ ]:
!pip install -q rouge_score

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
# ============================================================
# Imports and Initialization
# ============================================================

import os, math, json, random
from typing import List
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, BartForConditionalGeneration
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# -------------------------
# Config
# -------------------------
BACKBONE = "facebook/bart-base"
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128
DOC_SEP = "</s>"

ADAPTER_TYPE = "residual"       # choose: "residual" or "sparse"
N_ADAPTER_LAYERS = 3            # top-N layers to adapt (adapters)
ADAPTER_BOTTLENECK = 128
SPARSE_TOPK = 128
USE_FRACTION_FOR_SPARSE = False
SPARSITY_LAMBDA = 1e-5          # L1 regularization

TRAIN_SUBSET = 2000
VAL_SUBSET = 500
BATCH_SIZE = 32                    # or 16 or 8
EPOCHS = 5                         # or 8
LR = 3e-5
CHECKPOINT_DIR = "/content/bart_adapters_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

scorer_eval = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

In [ ]:
# ============================================================
# Utilities
# ============================================================
def flatten_abstracts_to_text(background: str, abstracts: List[str]):
    if background and background.strip():
        return background.strip() + " " + DOC_SEP + " " + (" " + DOC_SEP + " ").join(abstracts)
    return (" " + DOC_SEP + " ").join(abstracts)

def tokenize_texts(tokenizer, texts, max_length):
    return tokenizer(texts, truncation=True, padding="longest",
                     max_length=max_length, return_tensors="pt")

In [ ]:
# ============================================================
# Adapter Modules
# ============================================================
class ResidualAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64):
        super().__init__()
        # TODO: Define down-projection (hidden_size → bottleneck)
        # TODO: Define up-projection (bottleneck → hidden_size)
        # TODO: Initialize up layer weights and biases to zero
    def forward(self, x):
        # TODO: Apply down-projection and ReLU activation
        # TODO: Apply up-projection to get residual delta
        return x + delta

class SparseAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64, topk=128, use_fraction=False):
        super().__init__()
        # TODO: Define down-projection (hidden_size → bottleneck)
        # TODO: Define up-projection (bottleneck → hidden_size)
        # TODO: Initialize up layer weights and biases to zero
        # TODO: Define gating network for computing token importance scores
        # TODO: Store topk and use_fraction parameters

    def sample_gumbel(self, shape, device='cpu', eps=1e-20):
        U = torch.rand(shape, device=device)
        return -torch.log(-torch.log(U + eps) + eps)

    def relaxed_topk_mask(self, logits: torch.Tensor, k: int, tau: float = 1.0):
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        device = logits.device
        g = self.sample_gumbel(logits.shape, device=device)
        y = (logits + g) / tau
        probs = F.softmax(y, dim=-1)
        topk_idx = torch.topk(logits, k=k, dim=-1).indices
        hard_mask = torch.zeros_like(logits)
        hard_mask[0, topk_idx[0]] = 1.0
        mask = (hard_mask - probs).detach() + probs
        return mask.squeeze(0)

    def forward(self, x):
        # TODO: Compute gating logits from input
        # TODO: Build relaxed top-k masks for each batch element
        # TODO: Apply down-projection, activation, and up-projection
        # TODO: Multiply by mask and add residual correction to input
        # TODO: Return adapted output

    def l1_penalty(self):
        return torch.norm(self.down.weight, 1) + torch.norm(self.up.weight, 1)

# ============================================================
# Adapter Injection Helpers
# ============================================================
import types

def wrap_encoder_layer(layer, adapter):
    original_forward = layer.forward
    def forward_with_adapter(self, hidden_states, *args, **kwargs):
        outputs = original_forward(hidden_states, *args, **kwargs)
        if isinstance(outputs, torch.Tensor):
            adapted = adapter(outputs)
            return adapted
        elif isinstance(outputs, tuple):
            adapted = adapter(outputs[0])
            return (adapted,) + outputs[1:]
        else:
            return outputs
    layer.forward = types.MethodType(forward_with_adapter, layer)

def wrap_decoder_layer(layer, adapter):
    original_forward = layer.forward
    def forward_with_adapter(self, hidden_states, *args, **kwargs):
        outputs = original_forward(hidden_states, *args, **kwargs)
        if isinstance(outputs, tuple):
            hs = outputs[0]
            adapted = adapter(hs)
            return (adapted,) + outputs[1:]
        elif isinstance(outputs, torch.Tensor):
            adapted = adapter(outputs)
            return adapted
        else:
            return outputs
    layer.forward = types.MethodType(forward_with_adapter, layer)

In [ ]:
# ============================================================
# BART with In-Layer Adapters
# ============================================================
class BARTWithAdapters(nn.Module):
    def __init__(self, backbone=BACKBONE, adapter_type="residual",
                 n_adapter_layers=3, adapter_bottleneck=64, device=DEVICE):
        super().__init__()
        self.device = device
        print("Loading BART backbone:", backbone)
        # TODO: Load pre-trained BART backbone and tokenizer
        # TODO: Freeze all model parameters except output embeddings
        # TODO: Get encoder and decoder layers
        # TODO: Identify target layer indices for adapter insertion
        # TODO: Initialize encoder and decoder adapter modules (Residual/Sparse)
        # TODO: Wrap chosen layers with corresponding adapters
        # TODO: Set adapter parameters to require gradient updates
        # TODO: Move entire model to specified device

    def forward(self, input_ids, attention_mask,
                decoder_input_ids=None, decoder_attention_mask=None, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
            use_cache=False,
            return_dict=True
        )
       # TODO: Return dictionary with loss and logits

    def generate_summary(self, text: str, max_length=120, num_beams=4):
        # TODO: Tokenize input text
        # TODO: Generate summary using model.generate()
        # TODO: Decode generated token IDs into text

In [ ]:
# ============================================================
# Instantiate model
# ============================================================
model = BARTWithAdapters(backbone=BACKBONE, adapter_type=ADAPTER_TYPE,
                         n_adapter_layers=N_ADAPTER_LAYERS,
                         adapter_bottleneck=ADAPTER_BOTTLENECK, device=DEVICE)

# ============================================================
# Dataset — MS² (allenai/mslr2022, config ms2)
# ============================================================
print("Loading MS² dataset (allenai/mslr2022, config ms2)...")
ds = load_dataset("allenai/mslr2022", "ms2")
train_ds = ds["train"]
val_ds = ds["validation"]
print("Sizes:", len(train_ds), len(val_ds))

if TRAIN_SUBSET:
    train_ds = train_ds.select(range(min(TRAIN_SUBSET, len(train_ds))))
if VAL_SUBSET:
    val_ds = val_ds.select(range(min(VAL_SUBSET, len(val_ds))))

def hf_row_to_example(row):
    bg = row.get("background", "") or ""
    abstract_list = row.get("abstract", []) or []
    tgt = row.get("target", "") or row.get("summary", "") or ""
    return {"background": bg, "abstracts": abstract_list, "target": tgt}

train_examples = [hf_row_to_example(r) for r in train_ds]
val_examples = [hf_row_to_example(r) for r in val_ds]
print("Prepared: train", len(train_examples), "val", len(val_examples))

In [ ]:
# ============================================================
# Collate
# ============================================================
def collate_fn(batch):
    inputs, targets = [], []
    for ex in batch:
        txt = flatten_abstracts_to_text(ex["background"], ex["abstracts"])
        inputs.append(txt)
        targets.append(ex["target"])
    tok_inputs = tokenize_texts(model.tokenizer, inputs, MAX_INPUT_LENGTH)
    tok_targets = tokenize_texts(model.tokenizer, targets, MAX_TARGET_LENGTH)

    pad_id = model.tokenizer.pad_token_id
    labels = tok_targets["input_ids"].clone()
    labels[labels == pad_id] = -100
    labels = labels.long()
    decoder_input_ids = model.model.prepare_decoder_input_ids_from_labels(labels)
    decoder_attention_mask = (decoder_input_ids != pad_id).long()

    return {
        "input_ids": tok_inputs["input_ids"].to(DEVICE),
        "attention_mask": tok_inputs["attention_mask"].to(DEVICE),
        "decoder_input_ids": decoder_input_ids.to(DEVICE),
        "decoder_attention_mask": decoder_attention_mask.to(DEVICE),
        "labels": labels.to(DEVICE),
    }

train_loader = DataLoader(train_examples, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_examples, batch_size=1,
                        shuffle=False, collate_fn=collate_fn)

In [ ]:
# ============================================================
# Optimizer
# ============================================================
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

# ============================================================
# Training loop
# ============================================================
print("Starting training ...")
for epoch in range(EPOCHS):
    model.train()
    cum_loss, steps = 0.0, 0
    for batch in tqdm(train_loader, desc=f"Train ep{epoch}"):
        # TODO: Zero optimizer gradients
        # TODO: Forward pass through model to get loss

        # TODO: If adapter type is "sparse", compute L1 regularization penalty
        #       - Iterate through model modules
        #       - Accumulate L1 loss for all SparseAdapters
        #       - Add weighted penalty to main loss

        # TODO: Backpropagate gradients
        # TODO: Clip gradients to prevent exploding values
        # TODO: Perform optimizer step to update trainable parameters
        if steps % 20 == 0:
            print(f"Epoch {epoch} step {steps} loss {loss.item():.4f}")
    print(f"Epoch {epoch} avg loss {(cum_loss/steps) if steps>0 else 0.0:.4f}")
    torch.save(model.state_dict(),
               os.path.join(CHECKPOINT_DIR, f"bart_adapters_{ADAPTER_TYPE}_epoch{epoch}.pt"))

    # Quick validation (small subset)
    # TODO: Set model to evaluation mode
    # TODO: Select a small subset of validation examples
    # TODO: Disable gradient computation for evaluation
    # TODO: For each validation sample:
    #           - Flatten abstracts into text input
    #           - Generate summary using the model
    #           - Compute ROUGE metrics for evaluation
    # TODO: Compute and print average ROUGE-1, ROUGE-2, and ROUGE-L score

In [ ]:
# ============================================================
# Final Evaluation
# ============================================================
print("Running final evaluation ...")
model.eval()
# TODO: Model evaluation on validation set

torch.save(model.state_dict(),
           os.path.join(CHECKPOINT_DIR, f"bart_adapters_{ADAPTER_TYPE}_final.pt"))
print("Done......................................................")

## PART 4

In [ ]:
# ============================================================
# Full Fine-tuning: facebook/bart-base on MS² (allenai/mslr2022)
# ============================================================

# ============================================================
# Model and Tokenizer
# ============================================================
print("Loading BART model and tokenizer...")
model = BartForConditionalGeneration.from_pretrained(BACKBONE).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)

# ============================================================
# Dataset
# ============================================================
print("Loading MS² dataset (allenai/mslr2022, config ms2)...")
ds = load_dataset("allenai/mslr2022", "ms2")
train_ds, val_ds = ds["train"], ds["validation"]

if TRAIN_SUBSET:
    train_ds = train_ds.select(range(min(TRAIN_SUBSET, len(train_ds))))
if VAL_SUBSET:
    val_ds = val_ds.select(range(min(VAL_SUBSET, len(val_ds))))

def to_examples(split):
    out = []
    for r in split:
        out.append({
            "background": r.get("background", "") or "",
            "abstracts": r.get("abstract", []) or [],
            "target": r.get("target", "") or r.get("summary", "") or ""
        })
    return out

train_examples = to_examples(train_ds)
val_examples = to_examples(val_ds)
print("Prepared:", len(train_examples), "train,", len(val_examples), "val")

# ============================================================
# Collate Function
# ============================================================
def collate_fn(batch):
    inputs, targets = [], []
    for ex in batch:
        txt = flatten_abstracts_to_text(ex["background"], ex["abstracts"])
        inputs.append(txt)
        targets.append(ex["target"])
    tok_inputs = tokenize_texts(tokenizer, inputs, MAX_INPUT_LENGTH)
    tok_targets = tokenize_texts(tokenizer, targets, MAX_TARGET_LENGTH)
    pad_id = tokenizer.pad_token_id
    labels = tok_targets["input_ids"].clone()
    labels[labels == pad_id] = -100
    labels = labels.long()
    dec_in = model.prepare_decoder_input_ids_from_labels(labels)
    dec_mask = (dec_in != pad_id).long()
    return {
        "input_ids": tok_inputs["input_ids"].to(DEVICE),
        "attention_mask": tok_inputs["attention_mask"].to(DEVICE),
        "decoder_input_ids": dec_in.to(DEVICE),
        "decoder_attention_mask": dec_mask.to(DEVICE),
        "labels": labels.to(DEVICE)
    }

train_loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_examples, batch_size=1, shuffle=False, collate_fn=collate_fn)

# ============================================================
# Optimizer
# ============================================================
opt = torch.optim.AdamW(model.parameters(), lr=LR)

# ============================================================
# Training Loop
# ============================================================
print("Starting full fine-tuning...")
for epoch in range(EPOCHS):
    # TODO: complete training loop

    # quick validation check
    model.eval()
    with torch.no_grad():
        # TODO: complete quick validation

# ============================================================
# Final Evaluation
# ============================================================
print("\nRunning final evaluation...")
model.eval()
all_scores = []
# TODO: complete evaluation

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "bart_full_final.pt"))
print("Full fine-tuning complete.")

In [ ]:
#------------------------------------------------------------------------------
#------------------------------------------------------------------------------
#------------------------------------------------------------------------------